# 🎙️ ProXMax — English → Bangla ভিডিও ডাবিং

এই নোটবুকটি Google Drive-এর `Video/input` ফোল্ডারে থাকা **সব English ভিডিওকে** একে একে Bangla ডাবিং করে `Video/output` ফোল্ডারে রেখে দেবে।

**শুরু করার আগে (গুরুত্বপূর্ণ):**
1. উপরের মেনু থেকে **Runtime → Change runtime type → Hardware accelerator: T4 GPU** নির্বাচন করুন (GPU ছাড়া প্রসেসিং অনেক ধীর হবে)।
2. তারপর এই নোটবুকের সেলগুলো **উপর থেকে নিচে একে একে** রান করুন (প্রতিটি সেলের বাম পাশে ▶ বাটন)।

পুরো প্রক্রিয়া: `ভিডিও → অডিও আলাদা করা → English transcription (faster-whisper, GPU) → Bangla অনুবাদ → Bangla ভয়েস (edge-tts) → সময় মিলিয়ে sync → চূড়ান্ত ভিডিও`

## 📂 ধাপ ১: Google Drive mount করা

আপনার Google Drive-কে Colab-এর সাথে যুক্ত করা হচ্ছে, যাতে নোটবুকটি Drive-এর ভিডিও ফাইল পড়তে ও আউটপুট সেভ করতে পারে। অনুমতির (permission) পপ-আপ আসলে আপনার Google অ্যাকাউন্ট দিয়ে **Allow** করুন।

In [ ]:
from google.colab import drive

# Drive mount করা (/content/drive/MyDrive-এর নিচে পাওয়া যাবে)
drive.mount('/content/drive')
print('✅ Google Drive mount সম্পন্ন')

## 🗂️ ধাপ ২: Input/Output ফোল্ডার তৈরি ও যাচাই

Drive-এ `Video/input` ও `Video/output` ফোল্ডার না থাকলে এই সেল নিজে থেকেই তৈরি করে দেবে। তারপর input ফোল্ডারে যেসব ভিডিও আছে তাদের তালিকা দেখাবে — **ভিডিও আপলোড করা না থাকলে আগে Drive-এর `Video/input` ফোল্ডারে ভিডিও আপলোড করে নিন।**

In [ ]:
import os

INPUT_DIR = '/content/drive/MyDrive/Video/input'
OUTPUT_DIR = '/content/drive/MyDrive/Video/output'

# ফোল্ডার দুটি না থাকলে তৈরি করা (থাকলে কিছু হবে না)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('📥 Input ফোল্ডার :', INPUT_DIR)
print('📤 Output ফোল্ডার:', OUTPUT_DIR)

# input ফোল্ডারের ভিডিওগুলোর তালিকা দেখানো
videos = [f for f in sorted(os.listdir(INPUT_DIR))
          if f.lower().endswith(('.mp4', '.mkv', '.mov', '.avi'))]
if videos:
    print(f'\n🎬 মোট {len(videos)}টি ভিডিও পাওয়া গেছে — এগুলো একে একে ডাবিং হবে:')
    for i, name in enumerate(videos, start=1):
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, name)) / (1024 * 1024)
        print(f'  {i}. {name}  ({size_mb:.1f} MB)')
else:
    print('\n⚠️ Input ফোল্ডারে কোনো ভিডিও নেই!')
    print('   Google Drive ওপেন করে  Video/input  ফোল্ডারে আপনার English ভিডিও')
    print('   আপলোড করুন, তারপর এই সেলটি আবার রান করে যাচাই করুন।')

## 🖥️ ধাপ ৩: GPU যাচাই করা

ভিডিও ট্রান্সক্রিপশন GPU দিয়ে হয় বলে অনেক দ্রুত হয়। এই সেলের আউটপুটে **Tesla T4** দেখালে বুঝবেন GPU ঠিকঠাক পাওয়া গেছে। GPU না দেখালে Runtime → Change runtime type থেকে **T4 GPU** নির্বাচন করে আবার ধাপ ১ থেকে শুরু করুন।

In [ ]:
# GPU-এর তথ্য দেখানো (Tesla T4 দেখা উচিত)
!nvidia-smi

## ⬇️ ধাপ ৪: ProXMax রিপোজিটরি clone করা

GitHub থেকে ProXMax-এর কোড Colab-এ নামানো হচ্ছে। রিপো আগেই clone করা থাকলে সর্বশেষ আপডেটটি (`git pull`) টেনে নেওয়া হবে — তাই সেলটি একাধিকবার রান করা নিরাপদ।

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/redwan-primeAmax/ProXMax.git'
REPO_DIR = '/content/ProXMax'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('রিপো আগেই clone করা আছে — সর্বশেষ আপডেট টানা হচ্ছে...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    print('রিপো clone করা হচ্ছে...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('\n✅ বর্তমান ফোল্ডার:', os.getcwd())
print('ফাইলসমূহ:', sorted(os.listdir(REPO_DIR)))

## 📦 ধাপ ৫: Dependency ইনস্টল করা

`requirements.txt` থেকে সব পাইথন প্যাকেজ (faster-whisper, edge-tts ইত্যাদি) ইনস্টল হচ্ছে এবং **ffmpeg** আছে কিনা যাচাই করা হচ্ছে (না থাকলে ইনস্টল করা হবে)। প্রথমবারে এই সেলটি শেষ হতে কয়েক মিনিট লাগতে পারে।

> 💡 Whisper-এর মডেল (~3 GB) এখানে ডাউনলোড হয় না — মূল প্রোগ্রাম প্রথমবার রান করার সময় নিজে থেকেই ডাউনলোড করে নেবে।

In [ ]:
%cd /content/ProXMax
!pip install -q -r requirements.txt

In [ ]:
import glob
import importlib.metadata
import os
import shutil
import site
import subprocess

# ffmpeg না থাকলে apt দিয়ে ইনস্টল করা
if shutil.which('ffmpeg') is None:
    print('ffmpeg পাওয়া যায়নি — ইনস্টল করা হচ্ছে...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg',
                   shell=True, check=False)
print('ffmpeg পাথ:', shutil.which('ffmpeg'))

# GPU-র জন্য cuDNN/cuBLAS লাইব্রেরির পাথ ঠিক করে দেওয়া (Colab-এ দরকার হয়)
lib_dirs = []
for root in site.getsitepackages():
    lib_dirs += glob.glob(os.path.join(root, 'nvidia', '*', 'lib'))
if lib_dirs:
    old = os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + ([old] if old else []))

# মূল প্যাকেজগুলো ঠিকভাবে ইনস্টল হয়েছে কিনা যাচাই
for pkg in ['faster-whisper', 'edge-tts', 'deep-translator', 'pydub']:
    print(f'  ✓ {pkg} == {importlib.metadata.version(pkg)}')
print('\n✅ সব dependency প্রস্তুত!')

## ⚙️ ধাপ ৬: দ্রুত সেটিংস (ঐচ্ছিক)

না বদলালেও চলবে — ডিফল্ট মানেই সর্বোচ্চ accuracy পাওয়া যাবে।

- `WHISPER_MODEL`: `'large-v3'` = সেরা মান (ডিফল্ট) · `'medium'` = প্রায় দ্বিগুণ দ্রুত, মান সামান্য কম
- `TTS_VOICE`: `'bn-BD-NabanitaNeural'` = নারী কণ্ঠ (ডিফল্ট) · `'bn-BD-PradeepNeural'` = পুরুষ কণ্ঠ

আরও বিস্তারিত সেটিংস বদলাতে চাইলে রিপোর `config.yaml` ফাইলটি সম্পাদনা করতে পারেন।

In [ ]:
WHISPER_MODEL = 'large-v3'              # দ্রুত চাইলে: 'medium'
TTS_VOICE = 'bn-BD-NabanitaNeural'      # পুরুষ কণ্ঠ: 'bn-BD-PradeepNeural'

print('Whisper মডেল:', WHISPER_MODEL)
print('Bangla ভয়েস :', TTS_VOICE)

## ▶️ ধাপ ৭: মূল অ্যাপ রান করা (ডাবিং শুরু)

এই সেলটি রান করলেই `Video/input`-এর সব ভিডিও একটার পর একটা ডাবিং হয়ে `Video/output` ফোল্ডারে জমা হতে থাকবে। প্রতিটি ধাপের অগ্রগতি (progress) আউটপুটে দেখতে পাবেন।

> ⏳ প্রথম ভিডিওর সময় Whisper মডেল ডাউনলোড (~3 GB) হতে কিছু সময় লাগবে। মাঝপথে কোনো কারণে বন্ধ হয়ে গেলে শুধু এই সেলটি আবার রান করুন — যেসব ভিডিওর আউটপুট তৈরি হয়ে গেছে সেগুলো নিজে থেকেই বাদ যাবে (**resume** সুবিধা)।

In [ ]:
%cd /content/ProXMax

# মূল ডাবিং প্রোগ্রাম রান করা (input-এর সব ভিডিও একে একে প্রসেস হবে)
!python main.py --model {WHISPER_MODEL} --voice {TTS_VOICE}

## 🎉 ধাপ ৮: ফলাফল দেখা

ডাবিং শেষে `Video/output` ফোল্ডারের ফাইলগুলোর তালিকা দেখানো হচ্ছে। প্রতিটি ভিডিওর নামের শেষে `_bn` যোগ হয়ে `.mp4` ফাইল তৈরি হবে (যেমন `lecture1.mp4` → `lecture1_bn.mp4`), সাথে English ও Bangla সাবটাইটেল ফাইলও (`.srt`) থাকবে। ফাইলগুলো আপনার Google Drive থেকেই যেকোনো সময় ডাউনলোড/প্লে করতে পারবেন।

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/Video/output'
files = sorted(os.listdir(OUTPUT_DIR)) if os.path.isdir(OUTPUT_DIR) else []

if files:
    print(f'📤 Output ফোল্ডারে {len(files)}টি ফাইল আছে:')
    for name in files:
        size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, name)) / (1024 * 1024)
        print(f'  • {name}  ({size_mb:.1f} MB)')
    print('\n✅ কাজ শেষ! Google Drive-এর  Video/output  ফোল্ডার থেকে ভিডিওগুলো নিন।')
else:
    print('⚠️ Output ফোল্ডার খালি — উপরের ধাপ ৭-এর আউটপুটে কোনো error এসেছে কিনা দেখুন।')